In [1]:
# This script takes the predicted loads as input (Step 1 from 2-step approach)
# The power predictions from the models (tgt, Sarima, Xgboost, naive) are used as active Power
# The q_mvar var is added by using the case and bus specific scaling factor.
# Output: One csv for each model prediction
# Next step: input the csv files into datakit to retrieve the OPF solution for Step 2 of 2-step approach 

In [2]:
import os
import numpy as np
import pandas as pd
import importlib

#! CONFIG
#! If i want to do case118 i have to setup a new, datakit specific venv, as pypower needs numpy <2 and graphkit needs numpy >2.
CASE = 14                    # set IEEE case 
PARQUET = "forecasts.parquet"
OUT_ROOT = "input_for_opf"
ID_COL = "load_scenario_idx"

In [3]:
# cell 2 - helper: load IEEE base P/Q (try pypower, fallback)
def get_ieee_base(case_n: int):
    # try pypower
    try:
        api = importlib.import_module("pypower.api")
        fn = getattr(api, f"case{case_n}", None)
        if fn is not None:
            mpc = fn()
            P_base = mpc["bus"][:, 2].astype(float)
            Q_base = mpc["bus"][:, 3].astype(float)
            return P_base, Q_base
    except Exception:
        print("\n\n\nPYPOWER DOES NOT WORK!!!!!!\n\n\nPYPOWER DOES NOT WORK!!!!!!\n\n\nPYPOWER DOES NOT WORK!!!!!!\n\n\nPYPOWER DOES NOT WORK!!!!!!\n\n\n")
        pass

    #! fallback: embed known cases (add more if needed)
    if case_n == 14:
        P_BASE = np.array([0.0,21.7,94.2,47.8,7.6,11.2,0.0,0.0,29.5,9.0,3.5,6.1,13.5,14.9], dtype=float)
        Q_BASE = np.array([0.0,12.7,19.0,-3.9,1.6,7.5,0.0,0.0,16.6,5.8,1.8,1.6,5.8,5.0], dtype=float)
        return P_BASE, Q_BASE

    raise RuntimeError(f"Unsupported or unavailable case: {case_n}")


In [4]:
# cell 3 - load parquet, validate, derive ratio
if not os.path.exists(PARQUET):
    raise FileNotFoundError(PARQUET)

df = pd.read_parquet(PARQUET)

id_col = ID_COL


# get base P/Q and compute Q/P ratio 
P_base, Q_base = get_ieee_base(CASE)
P_base = np.asarray(P_base, dtype=float)
Q_base = np.asarray(Q_base, dtype=float)
ratio = np.zeros_like(P_base, dtype=float)
mask = P_base != 0.0
ratio[mask] = Q_base[mask] / P_base[mask]

FileNotFoundError: forecasts.parquet

In [ ]:
# cell 4 - transform & write one CSV per model column
exclude = {id_col, "bus_id", "horizon_step"}
model_cols = [c for c in df.columns if c not in exclude]

out_dir = os.path.join(OUT_ROOT, f"case{CASE}_ieee")
os.makedirs(out_dir, exist_ok=True)

bus_idx = df["bus_id"].to_numpy(dtype=int)
valid_mask = (bus_idx >= 0) & (bus_idx < len(ratio))
bus_idx_clamped = np.where(valid_mask, bus_idx, -1)

for col in model_cols:
    p_arr = pd.to_numeric(df[col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    q_arr = np.where(bus_idx_clamped >= 0, p_arr * ratio[bus_idx_clamped], 0.0)
    out = pd.DataFrame({
        "load_scenario": df[id_col].astype(int),
        "load": df["bus_id"].astype(int),
        "p_mw": p_arr,
        "q_mvar": q_arr
    })
    out_path = os.path.join(out_dir, f"{col}.csv")
    out.to_csv(out_path, index=False)
    print("wrote", out_path)

wrote input_for_opf\case14_ieee\true.csv
wrote input_for_opf\case14_ieee\xgb.csv
wrote input_for_opf\case14_ieee\snaive.csv
wrote input_for_opf\case14_ieee\tgt.csv
wrote input_for_opf\case14_ieee\sarima.csv


In [ ]:
import importlib
try:
    api = importlib.import_module("pypower.api")
    print("pypower import OK; has case14:", hasattr(api,"case14"), "has case118:", hasattr(api,"case118"))
except Exception as e:
    print("pypower import failed:", e)

pypower import failed: cannot import name 'in1d' from 'numpy' (d:\Data\studium\Master\MA_Code\thesis_env\Lib\site-packages\numpy\__init__.py)
